# Ylivertainen v2 — Clinical Association Pipeline

End-to-end workflow for finding statistically valid associations between **target outcomes**
and **predictor variables** in a clinical dataset.

This notebook drives six universal modules:

| Module                       | Purpose                                                    |
|------------------------------|------------------------------------------------------------|
| `schema_infer.py`            | Auto-classify each column (continuous, ordinal, …)         |
| `cleaning.py`                | Apply the schema, audit duplicates, derive new columns     |
| `dda.py`                     | Per-column descriptive stats + SVG plots                   |
| `missingness_resolution.py`  | Missing pattern analysis, flags, MICE multiple imputation  |
| `eda.py`                     | Univariate target × predictor screening (FDR-corrected)    |
| `inferential.py`             | Multivariable logistic regression with Rubin pooling       |

**Pipeline order**

```
load → infer schema → clean → DDA → missingness → derive new cols → DDA again
   → EDA screen → MICE impute → multivariable logistic (Rubin pool) → outputs
```

All outputs land under `output/<stage>/{figures,tables}/` as SVG and CSV.


## 0. Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from schema_infer import (infer_schema, print_schema_template, schema_summary,
                          export_schema_summary, ColSpec)
from cleaning import (apply_schema, audit_duplicates, export_cleaning_artifacts,
                      bin_numeric, bin_datetime, make_missing_flag, combine_categories)
from dda import run_dda, plot_distribution_by_year
from missingness_resolution import (analyze_missingness, add_missing_flags,
                                    mark_structural_missing, drop_rows,
                                    mice_impute, simple_impute)
from eda import screen_associations
from inferential import run_inferential

OUTPUT_ROOT = Path("output")

# Cohort: set to one calendar year, or None for all years in the file (see §1b).
ANALYSIS_YEARS: list[int] | None = None   # e.g. [2025]

#ANALYSIS_YEARS = [2025]
YEAR_COLUMN = "entry_year"
DDA_SKIP_COLS: list[str] = []      # auto-set to ["year"] when ANALYSIS_YEARS is set


## 1. Load your data

Change the path to point at your Excel/CSV file. The rest of the notebook is dataset-agnostic.


In [2]:
DATA_PATH = "Meningiomas PSKUS grants.xlsx"   # or "yourdata.csv"

if str(DATA_PATH).endswith(".csv"):
    df_raw = pd.read_csv(DATA_PATH, index_col="Nr.")
else:
    df_raw = pd.read_excel(DATA_PATH, index_col="Nr.")

print(f"Loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
df_raw.head(0)


Loaded: 398 rows × 38 columns


,Personas kods,Unnamed: 2,"Vecums, gadi","Dzimums, 0 - vīrietis\n1 - sieviete""","Histoloģija, 0 - nav\n1 - ir","WHO pakāpe (2021), 1 / 2 / 3","Progesterons, 0 - negatīvs\n1 - pozitīvs","Ki-67 (%), skaitlis, %","Smadzeņu parenhīmas invāzija, 0 - nav\n1 - ir","Nekroze histoloģiski, 0 - nav\n1 - ir",...,"Audzēja nekroze, 0 - nav\n1 - ir","Hemorāģiskas sastāvdaļas, 0 - nav\n1 - ir\n2 - nav skaidri izvērtējams","Kaule hiperostoze, 0 - nav\n1 - ir","Kaula invāzija (cortical destruction), 0 - nav\n1 - ir","Tumor Hyperintensity on DWI, 0 - nav\n1 - ir","Tumor Hyperintensity on T2, 0 - nav\n1 - ir","Tumor Hypointensity on T1, 0 - nav\n1 - ir","Sīnuss, 0 - neieaug\n1 - ieaug\n2 - ieaug un cauraug","Cauraug falx cerebri 0 - nav, 1 - ir",ADC map value
Nr.,,,,,,,,,,,,,,,,,,,,,


### 1a. (Optional) Rename columns to clean snake_case

If your source has long Latvian/Russian/free-text column names, rename them here.
Comment this cell out for new datasets where column names are already clean.


In [3]:
# Example for RPE study — edit/remove for other datasets:
# df_raw.columns = ['Gads', 'pk', 'Vecums', 'Pirmsoperācijas_PSA', 'lesion_1_MRI_PIRADS', ...]
# df_raw.columns = [c.strip() for c in df_raw.columns]

#🟧🟧🟧 comfort renaming
COLUMN_RENAME_MAP = {
    "Personas kods": "patient_code",
    "Unnamed: 2": "entry_year",

    "Vecums, gadi": "age",
    'Dzimums, 0 - vīrietis\n1 - sieviete"': "sex",
    "Histoloģija, 0 - nav\n1 - ir": "histology_available",
    "WHO pakāpe (2021), 1 / 2 / 3": "who_grade",
    "Progesterons, 0 - negatīvs\n1 - pozitīvs": "progesterone_pos",
    "Ki-67 (%), skaitlis, %": "ki67_pct",

    "Smadzeņu parenhīmas invāzija, 0 - nav\n1 - ir": "brain_invasion",
    "Nekroze histoloģiski, 0 - nav\n1 - ir": "hist_necrosis",

    "MRI izmeklējuma datums": "mri_date",
    "Puse, 1 - labā\n2 - kreisā\n3 - viduslīnija": "side",
    "Lokalizācija: skull base / non–skull base, 0 - non-skull base\n1 - skull base": "tumor_location",
    "Cik meningiomas?": "meningioma_count",

    "Max diametrs, skaitlis,cm": "max_diameter_cm",
    "Tilpums": "tumor_volume",
    "Pamatmodalitāte analīzei, 0 - MRI\n1 - CT\n3 - MRI+CT": "base_modality",
    "K/v i/v, 0 - nav\n1 - ir": "iv_contrast",
    "0 - primārs\n1 - recidīvs": "tumor_episode",

    "Audzēja robeža, 1 = gluda, \n2 = neregulāra": "tumor_margin",
    "Dural tail sign, 0 - nav\n1 - ir": "dural_tail",
    "Gredzenveida kontrastēšanās (tumor capsular enhancement), 0 - nav\n1 - ir": "capsular_enhancement",
    "Kontrastēšanās veids, 0 - homogēna\n1 - heterogēna": "heterogeneous_enhancement",

    "Perifokāla tūska, 0 - nav\n1 - ir": "perifocal_edema",
    "Perifokālas tūskas tilpums, cm2": "edema_volume_cm3",
    "Masas efekts, 0 - nav\n1 - ir": "mass_effect",

    "Audzēja kalcifikācija, 0 - nav\n1 - ir": "calcification",
    "Cistiskas komponentes, 0 - nav\n1 - ir": "cystic_component",
    "Audzēja nekroze, 0 - nav\n1 - ir": "necrosis",
    "Hemorāģiskas sastāvdaļas, 0 - nav\n1 - ir\n2 - nav skaidri izvērtējams ": "hemorrhage",

    "Kaule hiperostoze, 0 - nav\n1 - ir": "hyperostosis",
    "Kaula invāzija (cortical destruction), 0 - nav\n1 - ir": "cortical_destruction",

    "Tumor Hyperintensity on DWI, 0 - nav\n1 - ir": "dwi_hyperintensity",
    "Tumor Hyperintensity on T2, 0 - nav\n1 - ir": "t2_hyperintensity",
    "Tumor Hypointensity on T1, 0 - nav\n1 - ir": "t1_hypointensity",

    "Sīnuss, 0 - neieaug\n1 - ieaug\n2 - ieaug un cauraug": "sinus_invasion",
    "Cauraug falx cerebri 0 - nav, 1 - ir": "transfalcine_extension",
    "ADC map value": "adc_value",
    }

df_raw = df_raw.rename(columns=COLUMN_RENAME_MAP)
df = df_raw

#df = df_raw.reindex(['Gads', 'Biopsijas_veids'], axis=1)

df.head(0)

,patient_code,entry_year,age,sex,histology_available,who_grade,progesterone_pos,ki67_pct,brain_invasion,hist_necrosis,...,necrosis,hemorrhage,hyperostosis,cortical_destruction,dwi_hyperintensity,t2_hyperintensity,t1_hypointensity,sinus_invasion,transfalcine_extension,adc_value
Nr.,,,,,,,,,,,,,,,,,,,,,


### 1b. Cohort filter (optional)

Set `ANALYSIS_YEAR` in **§0 Setup** (e.g. `2025`), then run this cell after column rename.
Re-run the notebook from here through the report if you change the year.

In [4]:
if ANALYSIS_YEARS is not None and len(ANALYSIS_YEARS) == 0:
    raise ValueError("ANALYSIS_YEARS is []; use None for all years or e.g. [2025]")

if ANALYSIS_YEARS is not None:
    n_before = len(df_raw)
    df_raw = df_raw.loc[pd.to_numeric(df_raw[YEAR_COLUMN], errors="coerce").isin(ANALYSIS_YEARS)].copy()
    df = df_raw
    if df_raw.empty:
        raise ValueError(f"No rows with {YEAR_COLUMN} in {ANALYSIS_YEARS!r}")
    print(
        f"Cohort: {YEAR_COLUMN} in {ANALYSIS_YEARS} → "
        f"{len(df_raw)} rows (dropped {n_before - len(df_raw)})"
    )
else:
    DDA_SKIP_COLS = []
    years = sorted(pd.to_numeric(df_raw[YEAR_COLUMN], errors="coerce").dropna().astype(int).unique())
    print(f"Cohort: all years {list(years)} — {len(df_raw)} rows")

Cohort: all years [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)] — 398 rows


## 2. Infer schema and override

The engine auto-classifies every column. Inspect the printed dict, copy it into the
next cell, edit anything wrong (e.g. force `lesion_2_MRI_PIRADS` to `ordinal`,
mark `pk` as `id`, drop a junk column with `kind="skip"`).


In [5]:
schema = infer_schema(df_raw)
schema_summary(schema)


,column,kind,keep,ordered_levels,nulls,note
0,patient_code,id,True,None,None,
1,entry_year,datetime,True,None,None,
2,age,continuous,True,None,None,
3,sex,binary,True,None,None,
4,histology_available,ordinal,True,"[0.0, 1.0, 2.0]",None,
5,who_grade,nominal,True,None,None,
6,progesterone_pos,ordinal,True,"[0.0, 1.0, 2.0]",None,
7,ki67_pct,text,True,None,None,
8,brain_invasion,binary,True,None,None,
9,hist_necrosis,binary,True,None,None,


In [6]:
# Print a paste-back-able template; edit it in the next cell.
print_schema_template(schema);

schema_overrides = {
    'patient_code': ColSpec(name='patient_code', kind='id'),
    'entry_year': ColSpec(name='entry_year', kind='datetime'),
    'age': ColSpec(name='age', kind='continuous'),
    'sex': ColSpec(name='sex', kind='binary'),
    'histology_available': ColSpec(name='histology_available', kind='ordinal', ordered_levels=[0.0, 1.0, 2.0]),
    'who_grade': ColSpec(name='who_grade', kind='nominal'),
    'progesterone_pos': ColSpec(name='progesterone_pos', kind='ordinal', ordered_levels=[0.0, 1.0, 2.0]),
    'ki67_pct': ColSpec(name='ki67_pct', kind='text'),
    'brain_invasion': ColSpec(name='brain_invasion', kind='binary'),
    'hist_necrosis': ColSpec(name='hist_necrosis', kind='binary'),
    'mri_date': ColSpec(name='mri_date', kind='datetime'),
    'side': ColSpec(name='side', kind='ordinal', ordered_levels=[1.0, 2.0, 3.0]),
    'tumor_location': ColSpec(name='tumor_location', kind='ordinal', ordered_levels=[0.0, 1.0, 2.0]),
    'meningioma_count': ColSpec(name='meningi

### 2a. Paste the edited schema below

Take the printout from the cell above, paste it here, and adjust kinds/ordered_levels
as needed. Anything you don't override stays as inferred.

For RPE specifically, things to check:
- `pk` → `id`
- `preop_TNM_MDK`, `RPE_TNM` → `nominal`
- `risk_group` → `ordinal` with `ordered_levels=['zema','vidēja','augsta']`
- `biopsy_gleason_grade`, `RPE_grade`, PIRADS columns → `ordinal` with `[1,2,3,4,5]` (or `[0,1,2,3,4,5]`)
- `upgrade`, `upstage`, `downgrade`, `resection_lines_pos` → `binary`


In [ ]:
# Example override — adapt to your dataset!
schema_overrides = {
    'patient_code': ColSpec(name='patient_code', kind='id'),
    'entry_year': ColSpec(name='entry_year', kind='datetime'),
    'age': ColSpec(name='age', kind='continuous'),
    'sex': ColSpec(name='sex', kind='binary'),
    'histology_available': ColSpec(name='histology_available', kind='ordinal', ordered_levels=[0.0, 1.0, 2.0]),
    'who_grade': ColSpec(name='who_grade', kind='nominal'),
    'progesterone_pos': ColSpec(name='progesterone_pos', kind='ordinal', ordered_levels=[0.0, 1.0, 2.0]),
    'ki67_pct': ColSpec(name='ki67_pct', kind='text'),
    'brain_invasion': ColSpec(name='brain_invasion', kind='binary'),
    'hist_necrosis': ColSpec(name='hist_necrosis', kind='binary'),
    'mri_date': ColSpec(name='mri_date', kind='datetime'),
    'side': ColSpec(name='side', kind='ordinal', ordered_levels=[1.0, 2.0, 3.0]),
    'tumor_location': ColSpec(name='tumor_location', kind='ordinal', ordered_levels=[0.0, 1.0, 2.0]),
    'meningioma_count': ColSpec(name='meningioma_count', kind='ordinal', ordered_levels=[1.0, 2.0, 3.0, 4.0]),
    'max_diameter_cm': ColSpec(name='max_diameter_cm', kind='text'),
    'tumor_volume': ColSpec(name='tumor_volume', kind='text'),
    'base_modality': ColSpec(name='base_modality', kind='ordinal', ordered_levels=[0.0, 1.0, 3.0]),
    'iv_contrast': ColSpec(name='iv_contrast', kind='binary'),
    'tumor_episode': ColSpec(name='tumor_episode', kind='nominal'),
    'tumor_margin': ColSpec(name='tumor_margin', kind='ordinal', ordered_levels=[0.0, 1.0, 2.0]),
    'dural_tail': ColSpec(name='dural_tail', kind='binary'),
    'capsular_enhancement': ColSpec(name='capsular_enhancement', kind='binary'),
    'heterogeneous_enhancement': ColSpec(name='heterogeneous_enhancement', kind='binary'),
    'perifocal_edema': ColSpec(name='perifocal_edema', kind='binary'),
    'edema_volume_cm3': ColSpec(name='edema_volume_cm3', kind='text'),
    'mass_effect': ColSpec(name='mass_effect', kind='binary'),
    'calcification': ColSpec(name='calcification', kind='binary'),
    'cystic_component': ColSpec(name='cystic_component', kind='binary'),
    'necrosis': ColSpec(name='necrosis', kind='binary'),
    'hemorrhage': ColSpec(name='hemorrhage', kind='ordinal', ordered_levels=[0.0, 1.0, 2.0]),
    'hyperostosis': ColSpec(name='hyperostosis', kind='binary'),
    'cortical_destruction': ColSpec(name='cortical_destruction', kind='binary'),
    'dwi_hyperintensity': ColSpec(name='dwi_hyperintensity', kind='binary'),
    't2_hyperintensity': ColSpec(name='t2_hyperintensity', kind='binary'),
    't1_hypointensity': ColSpec(name='t1_hypointensity', kind='binary'),
    'sinus_invasion': ColSpec(name='sinus_invasion', kind='ordinal', ordered_levels=[0.0, 1.0, 2.0]),
    'transfalcine_extension': ColSpec(name='transfalcine_extension', kind='binary'),
    'adc_value': ColSpec(name='adc_value', kind='text'),
}
# merge overrides on top of inferred schema:
schema.update(schema_overrides)

schema_summary(schema)
export_schema_summary(schema, OUTPUT_ROOT)


PosixPath('output/schema/schema_summary.csv')

## 3. Apply schema → coerce dtypes, replacements, nulls

This is the only place dtypes are set. Downstream stages trust the schema.


In [8]:
schema_log = []
df = apply_schema(df_raw, schema, log=schema_log)
n_rows_after_schema = len(df)
df.dtypes

patient_code                         string
entry_year                   datetime64[ns]
age                                 float64
sex                                 boolean
histology_available                category
who_grade                          category
progesterone_pos                   category
ki67_pct                             string
brain_invasion                      boolean
hist_necrosis                       boolean
mri_date                     datetime64[us]
side                               category
tumor_location                     category
meningioma_count                   category
max_diameter_cm                      string
tumor_volume                         string
base_modality                      category
iv_contrast                         boolean
tumor_episode                      category
tumor_margin                       category
dural_tail                          boolean
capsular_enhancement                boolean
heterogeneous_enhancement       

## 4. Duplicate audit

Provide ID columns. The audit returns rows in duplicate groups and a cleaned frame.


In [9]:
ID_COLS = ['pk']   # edit for your dataset

dupes, df = audit_duplicates(df, id_cols=ID_COLS, include_first=True, drop=False)
print(f"Found {len(dupes)} duplicated rows (across {len(dupes)//2 if len(dupes) else 0}+ groups)")
dupes.head()


ValueError: id_cols not in dataframe: ['pk']

## 4b. Row removal — drop invalid records

After dtypes are coerced and duplicates audited, this is the place to permanently
remove rows that should not be in analysis at all (data-entry errors, ineligible
patients, impossible values). Every drop is **logged** so the methods section of
your paper can quote exact counts.

Common reasons:
- Out-of-range values (e.g. `vecums < 18`, `preop_PSA < 0`, `biopsy_to_RPE_days < 0`)
- Wrong cohort (e.g. patients without a primary RPE)
- Records missing critical identifiers
- Failed sanity checks against source records

Use `where=` for readable pandas-query strings, or `mask=` for arbitrary boolean
Series. Add as many calls as you need.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# Row removal — delete rows that should not be in the analysis at all.
#
# This is for rows that are STRUCTURALLY WRONG (out-of-cohort patient, data-
# entry errors, impossible values), NOT for rows with missing values
# (missingness is handled in section 6). Every call appends an entry to
# drop_log so you can report exact counts in your paper's methods section.
#
# Two ways to specify which rows to drop:
#   where='vecums < 18'                    → pandas query string (most readable)
#   mask=df['Gleason_grade_pēc_RPE'].isna() & ...   → arbitrary boolean Series (most powerful)
#
# Always provide a meaningful `reason` — it shows up in the audit log.
# ─────────────────────────────────────────────────────────────────────────

drop_log = []

# --- Examples below — edit / delete / uncomment for your dataset ---

df = drop_rows(df, mask=df['lesion_1_MRI_PIRADS'].isna(),
                reason='lesion_1_MRI_PIRADS == 0', log=drop_log)

# df = drop_rows(df, where='preop_PSA < 0',
#                reason='negative PSA = data entry error', log=drop_log)

# df = drop_rows(df, where='biopsy_to_RPE_days < 0',
#                reason='surgery before biopsy = entry error', log=drop_log)

# df = drop_rows(df, mask=df['Gleason_grade_pēc_RPE'].isna() & df['upgrade'].isna(),
#                reason='no histopathology recorded', log=drop_log)

# --- summary table ---
pd.DataFrame(drop_log) if drop_log else print('No rows dropped (uncomment examples above as needed)')

export_cleaning_artifacts(
    OUTPUT_ROOT,
    n_rows_raw=len(df_raw),
    n_rows_after_schema=n_rows_after_schema,
    n_rows_final=len(df),
    schema=schema,
    drop_log=drop_log,
    dupes=dupes,
    schema_log=schema_log,
)


{'summary': PosixPath('output/cleaning/cleaning_summary.csv'),
 'log': PosixPath('output/cleaning/cleaning_log.csv')}

## 5. DDA — first pass

Descriptive stats + SVG plots for every kept column.
Outputs land in `output/dda/`.


In [ ]:
dda_tables = run_dda(df, schema, output_root=OUTPUT_ROOT, skip_cols=DDA_SKIP_COLS)

print("\n--- continuous ---");  display(dda_tables['continuous'])
print("\n--- categorical ---"); display(dda_tables['categorical'])
print("\n--- binary ---");      display(dda_tables['binary'])
print("\n--- datetime ---");    display(dda_tables['datetime'])



--- continuous ---


,column,kind,n,n_unique,missing_pct,min,p_5th,median,mean,trimmed_mean,p_95th,max,mode,std,cv,iqr,skewness,kurtosis
0,Vecums,continuous,930,37,0.0,45.000000,53.000000,65.000000,64.450538,64.706989,74.000000,83.000000,67.00,6.555569,0.101715,9.000000,-0.362131,-0.147954
1,Pirmsoperācijas_PSA,continuous,930,586,0.0,0.313000,4.104500,8.000000,10.989413,9.158849,27.159500,100.000000,8.00,9.284227,0.844834,6.475000,3.494016,18.186023
2,no_biopsijas_līdz_RPE_dienas,continuous,930,281,0.0,0.000000,61.000000,118.000000,169.954839,128.494624,428.100000,3326.000000,113.00,216.476178,1.273728,75.000000,7.012083,71.483662
3,Priekšdziedzera_tilpums,continuous,930,313,0.0,2.000000,20.000000,39.075000,43.959011,41.292728,84.644500,155.000000,25.00,20.757808,0.472208,23.000000,1.529966,3.439026
4,PSA_blīvums,continuous,930,895,0.0,0.020579,0.076752,0.205852,0.298813,0.239945,0.830313,4.545455,0.25,0.323775,1.083538,0.198235,5.848351,56.625999



--- categorical ---


,column,kind,ordered,n,n_unique,missing_pct,first_mode,first_mode_pct,second_mode,second_mode_pct,rarest,max_class_imbalance,median_category,balance,entropy_bin
0,Gads,ordinal,True,930,5,0.00,2022,23.12,2023,22.47,2025,NaN,2022,0.9944,2.3089
1,lesion_1_MRI_PIRADS,ordinal,True,930,4,0.00,5,47.42,4,42.37,2,23.21,4,0.7227,1.4453
2,lesion_2_MRI_PIRADS,ordinal,True,232,4,75.05,4,59.48,3,26.72,2,13.80,4,0.7362,1.4724
3,lesion_3_MRI_PIRADS,ordinal,True,30,4,96.77,3,53.33,4,33.33,5,16.00,3,0.7539,1.5078
4,Pirmsoperācijas_TNM_MDK,nominal,False,930,15,0.00,T2cN0M0,36.24,T2N0M0,20.65,T2N1M0,NaN,NaN,0.6965,2.7213
5,EAU_riska_grupa,ordinal,True,930,3,0.00,vidēja,41.08,augsta,38.92,zema,2.05,vidēja,0.9600,1.5215
6,Gleason_grade_pēc_biopsijas,ordinal,True,930,5,0.00,2,41.08,1,39.89,5,15.92,2,0.7631,1.7717
7,Biopsijas_veids,nominal,False,930,2,0.00,transrektāla,69.68,transperineāla,30.32,transperineāla,2.30,NaN,0.8852,0.8852
8,Gleason_grade_pēc_RPE,ordinal,True,930,5,0.00,2,59.03,1,17.63,4,21.12,2,0.7140,1.6578
9,RPE_TNM,nominal,False,930,24,0.00,T3aN0M0,28.82,T2cN0M0,27.42,T4N0M0,268.00,NaN,0.6026,2.7628



--- binary ---


,column,kind,ordered,n,n_unique,missing_pct,first_mode,first_mode_pct,second_mode,second_mode_pct,rarest,max_class_imbalance,median_category,balance,entropy_bin
0,upgrade,binary,False,930,2,0.0,False,63.76,True,36.24,True,1.76,NaN,0.9446,0.9446
1,upstage,binary,False,930,2,0.0,False,59.03,True,40.97,True,1.44,NaN,0.9763,0.9763
2,downgrade,binary,False,930,2,0.0,False,87.20,True,12.80,True,6.82,NaN,0.5518,0.5518
3,Pozitīvas_rezekcijas_līnijas,binary,False,930,2,0.0,False,95.38,True,4.62,True,20.63,NaN,0.2702,0.2702



--- datetime ---


""


## 6. Missingness analysis

Per-column %, plus a Jaccard co-missingness heatmap so you can spot blocks of
columns that are missing together (often a data-entry-process artifact).


In [ ]:
missing_summary = analyze_missingness(df, output_root=OUTPUT_ROOT)
missing_summary

,column,n_missing,pct_missing
0,lesion_3_MRI_PIRADS,900,96.77
1,lesion_2_MRI_PIRADS,698,75.05
2,Gads,0,0.00
3,Gleason_grade_pēc_RPE,0,0.00
4,PSA_blīvums,0,0.00
5,RPE_TNM,0,0.00
6,downgrade,0,0.00
7,upstage,0,0.00
8,upgrade,0,0.00
9,Priekšdziedzera_tilpums,0,0.00


### 6b. Resolve missingness — tag structural vs MNAR

Not all `NaN` means the same thing. Before MICE imputes anything, classify each
column with missing values into one of three buckets:

| Bucket | Meaning | Action in this section |
|---|---|---|
| **Structural** | The value *does not exist* (e.g. `lesion_2_MRI_PIRADS` is NaN because the MRI showed only one lesion) | `mark_structural_missing(...)` — derives count + max features, flips originals to `kind='skip'` so MICE never touches them |
| **MNAR** | Missingness depends on the unobserved value itself (e.g. PSA not measured because the clinician judged it unnecessary) | `add_missing_flags(...)` in section 6c — adds an explicit `<col>_missing` flag, then imputes normally |
| **MAR** | Missingness depends only on *observed* variables | No special handling — MICE handles it correctly out of the box |

**Rule of thumb.** Ask: *"If this patient were re-examined today with perfect technique,
would a value exist?"* If **no** → structural. If **yes** → MAR/MNAR.

The structural step **must come before** MICE, because MICE will happily fabricate
PIRADS scores for non-existent lesions otherwise.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# STRUCTURAL_GROUPS — configure one entry per "slot family" in your data.
#
# A "slot family" is a set of columns that hold OPTIONAL repeats of the same
# clinical thing (lesion 1 / 2 / 3, tumour 1 / 2, biopsy core 1 / 2 / 3 …).
# NaN in slot 2 or 3 doesn't mean "we forgot to measure" — it means "that slot
# doesn't exist for this patient". Imputing it would invent findings.
#
# Each entry takes these keys:
#
#   'cols'         : list of all slot columns in the family (INCLUDE slot 1 here,
#                    so the count feature reflects the true number of slots).
#
#   'derive_count' : True  → create <group_name> = number of non-null slots.
#                    Use for "how many lesions did this patient have?"
#
#   'derive_max'   : True  → create <group_name>_max = max value across slots.
#                    Clinically the "dominant" lesion's PIRADS. Only set True
#                    when the slot columns are ORDINAL or NUMERIC.
#
#   'count_levels' : optional ordering for the count feature, e.g. [0,1,2,3].
#                    Sets the ordinal categories so charts/tables are ordered.
#
#   'max_levels'   : optional ordering for the max feature, e.g. [1,2,3,4,5].
#
#   'skip_after'   : list of columns to mark kind='skip' AFTER deriving features.
#                    IMPORTANT: usually leave the PRIMARY slot (lesion_1) OUT of
#                    this list — its value is real, not structural, and it stays
#                    as a normal predictor. Only skip the optional repeats.
#
# Effect: skipped columns are excluded from MICE imputation, EDA screening,
# the multivariable model, and the DDA second pass. They stay in the dataframe
# (you can still inspect them) but no statistic touches them.
# ─────────────────────────────────────────────────────────────────────────

STRUCTURAL_GROUPS = {
    # --- Example for RPE: 3 MRI lesion slots ---
     'n_lesions_MRI': {
         'cols':         ['lesion_1_MRI_PIRADS',
                          'lesion_2_MRI_PIRADS',
                          'lesion_3_MRI_PIRADS'],
         'derive_count': True,
         'derive_max':   True,
         'count_levels': [1, 2, 3],
         'max_levels':   [1, 2, 3, 4, 5],
         'skip_after':   ['lesion_2_MRI_PIRADS',   # ← optional slots only
                          'lesion_3_MRI_PIRADS'],  # ← lesion_1 stays as predictor
     },
}

if STRUCTURAL_GROUPS:
    df = mark_structural_missing(df, schema, STRUCTURAL_GROUPS)
    new_cols = [g for g in STRUCTURAL_GROUPS] + [f'{g}_max' for g in STRUCTURAL_GROUPS]
    print('Derived structural features:', [c for c in new_cols if c in df.columns])
    print('Now marked kind=\'skip\' (excluded from MICE / EDA / inferential):',
          [c for c, sp in schema.items() if sp.kind == 'skip'])
else:
    print('No structural-missing groups configured.')
    print('If your dataset has NaN that means "this slot does not exist",')
    print('edit STRUCTURAL_GROUPS above before running MICE in section 11.')


Derived structural features: ['n_lesions_MRI', 'n_lesions_MRI_max']
Now marked kind='skip' (excluded from MICE / EDA / inferential): ['lesion_2_MRI_PIRADS', 'lesion_3_MRI_PIRADS']


In [ ]:
# Sanity check — verify the new derived columns look right
derived = [g for g in STRUCTURAL_GROUPS] + [f'{g}_max' for g in STRUCTURAL_GROUPS]
derived = [c for c in derived if c in df.columns]
if derived:
    display(df[derived].describe(include='all'))
    for c in derived:
        print(c, df[c].value_counts(dropna=False).to_dict())


,n_lesions_MRI,n_lesions_MRI_max
count,930.0,930.000000
mean,1.28172,4.354839
std,0.516873,0.714981
min,1.0,2.000000
25%,1.0,4.000000
50%,1.0,4.000000
75%,1.0,5.000000
max,3.0,5.000000


n_lesions_MRI {np.int64(1): 698, np.int64(2): 202, np.int64(3): 30}
n_lesions_MRI_max {5.0: 443, 4.0: 392, 3.0: 77, 2.0: 18}


### 6c. Add MNAR missingness flags

For columns where the missingness itself carries information (true MNAR — e.g.
PSA not measured *because* risk looked low), add an explicit boolean flag so
the model can use 'was-it-measured' as a predictor. The schema is updated
automatically — no need to register the flag columns yourself.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# MNAR_COLS — list ONLY columns that meet ALL THREE criteria:
#
#   1. The value EXISTS in reality (it's not structurally absent — those went
#      into STRUCTURAL_GROUPS in section 6b).
#   2. The value is sometimes NOT recorded.
#   3. The reason it wasn't recorded is plausibly TIED TO THE VALUE ITSELF
#      (e.g. PSA not measured BECAUSE the clinician thought the patient was
#      low-risk → low-risk patients are systematically missing → MNAR).
#
# If missingness is purely due to data-entry chaos / random clerical loss,
# that's MAR — leave the column OUT of MNAR_COLS; MICE handles it correctly
# without a flag.
#
# Effect: for each column listed here, a new boolean column <col>_missing is
# added (True where the original was NaN) and registered in the schema as a
# binary predictor. The multivariable model can then use the FACT of missing-
# ness as its own predictor, separately from the imputed value.
# ─────────────────────────────────────────────────────────────────────────

MNAR_COLS = []   # e.g. ['Pirmsoperācijas_PSA', 'PSA_blīvums']

df = add_missing_flags(df, MNAR_COLS, schema=schema)
df.filter(like='_missing').head()


""
0
2
3
4
5


## 7. Derive new columns (vecums bins, time bins, PSA categories…)

Use the helpers below freely. Any new column you add **must** also be added to the
schema so DDA/EDA/inferential will analyze it.


In [ ]:
old_df_cols = df.columns.copy()

# ── 1. biopsy → RPE, in months ───────────────────────────────────────────
if 'no_biopsijas_līdz_RPE_dienas' in df.columns:
    df['no_biopsijas_līdz_RPE_mēneši'] = bin_numeric(
        df['no_biopsijas_līdz_RPE_dienas'],
        bins=[-np.inf, 90, 180, 270, 365, np.inf],
        labels=['<3 mo', '3-6 mo', '6-9 mo', '9-12 mo', '>12 mo'],
    )
    schema['no_biopsijas_līdz_RPE_mēneši'] = ColSpec(
        name='no_biopsijas_līdz_RPE_mēneši', kind='ordinal',
        ordered_levels=['<3 mo', '3-6 mo', '6-9 mo', '9-12 mo', '>12 mo'],
    )

# ── 2. PSA density, clinical 0.15 cutoff ─────────────────────────────────
if 'PSA_blīvums' in df.columns:
    df['PSA_blīvums_grupēts'] = bin_numeric(
        df['PSA_blīvums'],
        bins=[-np.inf, 0.15, np.inf],
        labels=['<0.15', '≥0.15'],
    )
    schema['PSA_blīvums_grupēts'] = ColSpec(
        name='PSA_blīvums_grupēts', kind='ordinal',
        ordered_levels=['<0.15', '≥0.15'],
    )

# ── 3. Prostate volume (ml) ──────────────────────────────────────────────
if 'Priekšdziedzera_tilpums' in df.columns:
    df['Priekšdziedzera_tilpums_grupēts'] = bin_numeric(
        df['Priekšdziedzera_tilpums'],
        bins=[-np.inf, 30, 50, 80, np.inf],
        labels=['<30 ml', '30-50 ml', '50-80 ml', '>80 ml'],
    )
    schema['Priekšdziedzera_tilpums_grupēts'] = ColSpec(
        name='Priekšdziedzera_tilpums_grupēts', kind='ordinal',
        ordered_levels=['<30 ml', '30-50 ml', '50-80 ml', '>80 ml'],
    )

# ── 4. pirmsoperācijas PSA (ng/ml) — standard NCCN risk strata ─────────────────────
if 'Pirmsoperācijas_PSA' in df.columns:
    df['Pirmsoperācijas_PSA_grupēts'] = bin_numeric(
        df['Pirmsoperācijas_PSA'],
        bins=[-np.inf, 4, 10, 20, np.inf],
        labels=['<4', '4-10', '10-20', '>20'],
    )
    schema['Pirmsoperācijas_PSA_grupēts'] = ColSpec(
        name='Pirmsoperācijas_PSA_grupēts', kind='ordinal',
        ordered_levels=['<4', '4-10', '10-20', '>20'],
    )

# ── 5. vecums (years) ───────────────────────────────────────────────────────
if 'Vecums' in df.columns:
    df['Vecums_grupēts'] = bin_numeric(
        df['Vecums'],
        bins=[-np.inf, 60, 70, 80, np.inf],
        labels=['<60', '60-69', '70-79', '≥80'],
    )
    schema['Vecums_grupēts'] = ColSpec(
        name='Vecums_grupēts', kind='ordinal',
        ordered_levels=['<60', '60-69', '70-79', '≥80'],
    )

# ── 6. PIRADS max — needs max_PIRADS column first ────────────────────────
# `mark_structural_missing` already created `n_lesions_MRI_max` (= max PIRADS).
# Rename for clarity if not done already.
if 'n_lesions_MRI_max' in df.columns and 'max_PIRADS' not in df.columns:
    df = df.rename(columns={'n_lesions_MRI_max': 'max_PIRADS'})
    if 'n_lesions_MRI_max' in schema:
        spec = schema.pop('n_lesions_MRI_max')
        spec.name = 'max_PIRADS'
        schema['max_PIRADS'] = spec

if 'max_PIRADS' in df.columns:
    df['max_PIRADS'] = bin_numeric(
        df['max_PIRADS'],
        bins=[-np.inf, 3, 4, np.inf],
        labels=['≤3', '4', '5'],
    )
    schema['max_PIRADS'] = ColSpec(
        name='max_PIRADS', kind='ordinal',
        ordered_levels=['≤3', '4', '5'],
    )

# -- 7. Clinically significant upgrade (specific GG shifts) --
# Grade codes 1–5 = GG1–GG5. Event = any of these biopsy → RPE pairs:
# GG1→GG3, GG1→GG4, GG2→GG3, GG2→GG4, GG2→GG5
if 'Gleason_grade_pēc_biopsijas' in df.columns and 'Gleason_grade_pēc_RPE' in df.columns:
    bx = pd.to_numeric(df['Gleason_grade_pēc_biopsijas'], errors='coerce')
    rpe = pd.to_numeric(df['Gleason_grade_pēc_RPE'], errors='coerce')
    _CLINICAL_GG_UPGRADE = {(1, 3), (1, 4), (1, 5), (2, 3), (2, 4), (2, 5)}
    _pair_key = (
        bx.astype('Int64').astype(str) + '→' + rpe.astype('Int64').astype(str)
    )
    _valid_keys = {f"{b}→{r}" for b, r in _CLINICAL_GG_UPGRADE}
    out = pd.Series(pd.NA, index=df.index, dtype='boolean')
    ok = bx.notna() & rpe.notna()
    out.loc[ok] = _pair_key.loc[ok].isin(_valid_keys)
    df['Klīniski_nozīmīgs_upgrade'] = out

    schema['Klīniski_nozīmīgs_upgrade'] = ColSpec(
        name='Klīniski_nozīmīgs_upgrade',
        kind='binary',
        note='GG1→3/4/5 or GG2→3/4/5 (grade codes 1-5)',
    )

if 'Pirmsoperācijas_TNM_MDK' in df.columns and 'RPE_TNM' in df.columns:
    _t = lambda s: pd.to_numeric(s.astype(str).str.extract(r'^T(\d)')[0], errors='coerce')
    bx, rpe = _t(df['Pirmsoperācijas_TNM_MDK']), _t(df['RPE_TNM'])
    out = pd.Series(pd.NA, index=df.index, dtype='boolean')
    ok = bx.notna() & rpe.notna()
    out.loc[ok] = bx[ok].isin([1, 2]) & rpe[ok].isin([3, 4, 5])
    df['Klīniski_nozīmīgs_upstage'] = out

    schema['Klīniski_nozīmīgs_upstage'] = ColSpec(
        name='Klīniski_nozīmīgs_upstage',
        kind='binary',
        note='T1→T3/4/5 or T2→3/4/5 (TNM codes where T1-5)',
    )

if 'downgrade' in df.columns and 'upgrade' in df.columns:
    df['Konkordance'] = ~df.upgrade & ~df.downgrade
    
    schema['Konkordance'] = ColSpec(
        name='Konkordance',
        kind='binary')

new_df_cols = df.columns.difference(old_df_cols)
df[new_df_cols].head()

,Klīniski_nozīmīgs_upgrade,Klīniski_nozīmīgs_upstage,Konkordance,PSA_blīvums_grupēts,Pirmsoperācijas_PSA_grupēts,Priekšdziedzera_tilpums_grupēts,Vecums_grupēts,max_PIRADS,no_biopsijas_līdz_RPE_mēneši
0,False,False,True,≥0.15,10-20,30-50 ml,<60,5,6-9 mo
2,False,False,True,<0.15,4-10,30-50 ml,<60,4,3-6 mo
3,False,False,True,<0.15,4-10,50-80 ml,70-79,5,3-6 mo
4,False,False,True,<0.15,4-10,50-80 ml,60-69,5,6-9 mo
5,False,False,False,<0.15,4-10,30-50 ml,60-69,5,3-6 mo


## 8. DDA — second pass (with derived columns)

Re-run DDA so the new columns get their own plots and stats.


In [ ]:
dda_tables = run_dda(df, schema, output_root=OUTPUT_ROOT, skip_cols=DDA_SKIP_COLS)
dda_tables['categorical']

,column,kind,ordered,n,n_unique,missing_pct,first_mode,first_mode_pct,second_mode,second_mode_pct,rarest,max_class_imbalance,median_category,balance,entropy_bin
0,Gads,ordinal,True,930,5,0.0,2022,23.12,2023,22.47,2025,NaN,2022,0.9944,2.3089
1,lesion_1_MRI_PIRADS,ordinal,True,930,4,0.0,5,47.42,4,42.37,2,23.21,4,0.7227,1.4453
2,Pirmsoperācijas_TNM_MDK,nominal,False,930,15,0.0,T2cN0M0,36.24,T2N0M0,20.65,T2N1M0,NaN,NaN,0.6965,2.7213
3,EAU_riska_grupa,ordinal,True,930,3,0.0,vidēja,41.08,augsta,38.92,zema,2.05,vidēja,0.9600,1.5215
4,Gleason_grade_pēc_biopsijas,ordinal,True,930,5,0.0,2,41.08,1,39.89,5,15.92,2,0.7631,1.7717
5,Biopsijas_veids,nominal,False,930,2,0.0,transrektāla,69.68,transperineāla,30.32,transperineāla,2.30,NaN,0.8852,0.8852
6,Gleason_grade_pēc_RPE,ordinal,True,930,5,0.0,2,59.03,1,17.63,4,21.12,2,0.7140,1.6578
7,RPE_TNM,nominal,False,930,24,0.0,T3aN0M0,28.82,T2cN0M0,27.42,T4N0M0,268.00,NaN,0.6026,2.7628
8,n_lesions_MRI,ordinal,True,930,3,0.0,1,75.05,2,21.72,3,23.27,1.0,0.5988,0.9490
9,no_biopsijas_līdz_RPE_mēneši,ordinal,True,930,5,0.0,3-6 mo,52.90,<3 mo,25.59,9-12 mo,11.44,3-6 mo,0.7703,1.7887


In [ ]:
df.columns

Index(['Gads', 'pk', 'Vecums', 'Pirmsoperācijas_PSA', 'lesion_1_MRI_PIRADS',
       'lesion_2_MRI_PIRADS', 'lesion_3_MRI_PIRADS', 'Pirmsoperācijas_TNM_MDK',
       'EAU_riska_grupa', 'Gleason_grade_pēc_biopsijas', 'Biopsijas_veids',
       'Gleason_grade_pēc_RPE', 'no_biopsijas_līdz_RPE_dienas',
       'Priekšdziedzera_tilpums', 'upgrade', 'upstage', 'downgrade', 'RPE_TNM',
       'PSA_blīvums', 'Pozitīvas_rezekcijas_līnijas', 'n_lesions_MRI',
       'max_PIRADS', 'no_biopsijas_līdz_RPE_mēneši', 'PSA_blīvums_grupēts',
       'Priekšdziedzera_tilpums_grupēts', 'Pirmsoperācijas_PSA_grupēts',
       'Vecums_grupēts', 'Klīniski_nozīmīgs_upgrade',
       'Klīniski_nozīmīgs_upstage', 'Konkordance'],
      dtype='str')

## 9. Configure targets and predictors

This is the only place outcome variables and candidate predictors are declared.


In [ ]:
EDA_TARGETS = ['upgrade', 'upstage', 'downgrade', 'Konkordance',
        'Klīniski_nozīmīgs_upgrade', 'Klīniski_nozīmīgs_upstage',
        'Pozitīvas_rezekcijas_līnijas']
EDA_PREDICTORS = ['Vecums', 'Pirmsoperācijas_PSA', 'lesion_1_MRI_PIRADS',
       'lesion_2_MRI_PIRADS', 'lesion_3_MRI_PIRADS', 'Pirmsoperācijas_TNM_MDK',
       'EAU_riska_grupa', 'Gleason_grade_pēc_biopsijas', 'Biopsijas_veids',
       'Gleason_grade_pēc_RPE', 'no_biopsijas_līdz_RPE_dienas',
       'Priekšdziedzera_tilpums',
       'PSA_blīvums', 'n_lesions_MRI',
       'max_PIRADS', 'no_biopsijas_līdz_RPE_mēneši', 'PSA_blīvums_grupēts',
       'Priekšdziedzera_tilpums_grupēts', 'Pirmsoperācijas_PSA_grupēts',
       'Vecums_grupēts']

INFERENTIAL_TARGETS = ['upgrade', 'upstage', 'downgrade', 'Konkordance', 
        'Klīniski_nozīmīgs_upgrade', 'Klīniski_nozīmīgs_upstage',
        'Pozitīvas_rezekcijas_līnijas']
INFERENTIAL_PREDICTORS = [
    'Vecums_grupēts',
    'Biopsijas_veids',
    'PSA_blīvums_grupēts',
    'lesion_1_MRI_PIRADS',
    'n_lesions_MRI',
    ]

#TARGETS = ['no_biopsijas_līdz_RPE_mēneši']
#PREDICTORS = ['no_biopsijas_līdz_RPE_mēneši']

EDA_PREDICTORS = [c for c in EDA_PREDICTORS if c in df.columns]  # drop any missing names
INFERENTIAL_PREDICTORS = [c for c in INFERENTIAL_PREDICTORS if c in df.columns]  # drop any missing names

# Binary targets only: which value counts as the event (positive class).
EDA_POSITIVE_CLASS = {t: True for t in EDA_TARGETS if t in (
    'upgrade', 'upstage', 'downgrade', 'Pozitīvas_rezekcijas_līnijas',
    'Klīniski_nozīmīgs_upgrade', 'Konkordance')}
INFERENTIAL_POSITIVE_CLASS = {t: True for t in INFERENTIAL_TARGETS if t in (
    'upgrade', 'upstage', 'downgrade', 'Pozitīvas_rezekcijas_līnijas',
    'Klīniski_nozīmīgs_upgrade', 'Konkordance')}

## 10. EDA — univariate screening

Targets can be **binary**, **continuous**, **ordinal**, or **nominal** (from schema). The test depends on both outcome and predictor types — e.g. ordinal outcome × nominal predictor → χ²; continuous outcome × nominal predictor → Kruskal–Wallis.

| target kind   | continuous / count predictor | ordinal predictor | nominal / binary predictor |
|---------------|------------------------------|-------------------|----------------------------|
| binary        | Mann–Whitney U               | Spearman ρ        | χ² / Fisher                |
| continuous    | Spearman ρ                   | Spearman ρ        | Kruskal–Wallis             |
| ordinal       | Spearman ρ                   | Spearman ρ        | χ²                         |
| nominal       | Kruskal–Wallis               | χ²                | χ²                         |

`POSITIVE_CLASS` applies only to **binary** targets. Multivariable logistic (§11) remains **binary outcomes only**.

Per-target p-values are corrected with **Benjamini–Hochberg FDR**.


In [ ]:
assoc = screen_associations(
    df, schema,
    targets=EDA_TARGETS,
    predictors=EDA_PREDICTORS,
    positive_class=EDA_POSITIVE_CLASS,
    fdr_alpha=0.05,
    output_root=OUTPUT_ROOT,
)

assoc[assoc['fdr_significant']]


,target,target_kind,predictor,kind,test,stat,p,p_fdr,fdr_significant,effect,effect_label,n_used,positive_class
0,Klīniski_nozīmīgs_upgrade,binary,Gleason_grade_pēc_RPE,ordinal,spearman,0.496002,6.557280e-59,1.311456e-57,True,0.496002,spearman_rho,930,True
1,Klīniski_nozīmīgs_upgrade,binary,lesion_1_MRI_PIRADS,ordinal,spearman,0.105586,1.261366e-03,1.261366e-02,True,0.105586,spearman_rho,930,True
2,Klīniski_nozīmīgs_upgrade,binary,lesion_2_MRI_PIRADS,skip,chi2,14.843018,1.955836e-03,1.303890e-02,True,0.252940,cramers_v,232,True
20,Klīniski_nozīmīgs_upstage,binary,Pirmsoperācijas_TNM_MDK,nominal,chi2,80.119192,2.689126e-11,5.378252e-10,True,0.293513,cramers_v,930,True
21,Klīniski_nozīmīgs_upstage,binary,Gleason_grade_pēc_RPE,ordinal,spearman,0.141581,1.466332e-05,1.466332e-04,True,0.141581,spearman_rho,930,True
22,Klīniski_nozīmīgs_upstage,binary,Pirmsoperācijas_PSA,continuous,mann_whitney_u,110247.500000,2.098535e-04,1.399023e-03,True,-0.149076,rank_biserial_r,930,True
23,Klīniski_nozīmīgs_upstage,binary,PSA_blīvums,continuous,mann_whitney_u,108668.000000,9.756137e-04,4.878068e-03,True,-0.132613,rank_biserial_r,930,True
24,Klīniski_nozīmīgs_upstage,binary,Pirmsoperācijas_PSA_grupēts,ordinal,spearman,0.095688,3.490613e-03,1.394632e-02,True,0.095688,spearman_rho,930,True
25,Klīniski_nozīmīgs_upstage,binary,PSA_blīvums_grupēts,ordinal,spearman,0.093832,4.183897e-03,1.394632e-02,True,0.093832,spearman_rho,930,True
40,Konkordance,binary,Gleason_grade_pēc_RPE,ordinal,spearman,-0.256577,1.904129e-15,3.808259e-14,True,-0.256577,spearman_rho,930,True


In [ ]:
# Full table
#assoc


## 11. Multiple imputation (MICE)

We generate **m=10** imputed datasets via sklearn's IterativeImputer
(RandomForest estimator, separate random seed per imputation).
The pooled inferential stage applies Rubin's rules over these 10 fits.

For a quick screening run set `m=3`. For publication use `m≥10`.


In [ ]:
M = 10  # number of imputations; reduce to 3 for fast iteration

imputed_frames = mice_impute(df, schema, m=M, max_iter=10,
                             random_state=42, output_root=OUTPUT_ROOT)
print(f"Generated {len(imputed_frames)} imputed frames")
print("NaN count in first imputed frame:", imputed_frames[0].isna().sum().sum())


Generated 10 imputed frames
NaN count in first imputed frame: 1598


## 12. Multivariable logistic regression (Rubin-pooled)

For each target:

1. Build design matrix (continuous z-scored, ordinal kept as codes, nominal one-hot).
2. Iteratively drop predictors with **VIF > 5** to handle collinearity.
3. Fit logistic regression on each of the m imputed frames.
4. Pool coefficients with **Rubin's rules** (Barnard–Rubin df).
5. Report adjusted OR with 95% CI and pooled p-value.
6. Save a forest plot SVG per target.


In [ ]:
inf_results = run_inferential(
    imputed_frames, schema,
    targets=INFERENTIAL_TARGETS,
    predictors=INFERENTIAL_PREDICTORS,
    positive_class=INFERENTIAL_POSITIVE_CLASS,
    vif_threshold=5.0,
    output_root=OUTPUT_ROOT,
)
#inf_results

In [ ]:
# Significant adjusted predictors per target (binary outcomes only)
if inf_results.empty:
    print("No multivariable results — §11 logistic needs a binary TARGETS entry "
          "(e.g. upgrade). Ordinal/continuous targets are EDA-only (§10).")
else:
    inf_results[(inf_results["p"] < 0.05) & inf_results["or"].notna()]


## 13. **REPORT**


In [ ]:
from report import build_report, ReportConfig, write_html
from pathlib import Path

# Primary exposure — gets a dedicated stats + figures block in Final conclusion
FOCUS_PREDICTOR = "Biopsijas_veids"
# Highlight this biopsy route in Variable of interest (reference in regression).
FOCUS_REFERENCE_LEVEL = "transperineāla"

_report_title = (
    "Transrectal Versus Transperineal Prostate Biopsy: "
    "Impact on Radical Prostatectomy Outcomes"
)
if ANALYSIS_YEARS is not None:
    _report_title += f" ({ANALYSIS_YEARS} cohort)"

# Two-panel focus figure (overall counts + within-year shares) when cohort spans years.
if ANALYSIS_YEARS is None and FOCUS_PREDICTOR and YEAR_COLUMN in df.columns:
    _by_year = plot_distribution_by_year(
        df, FOCUS_PREDICTOR, YEAR_COLUMN, OUTPUT_ROOT / "dda" / "figures"
    )
    if _by_year:
        print(f"Focus by-year figure: {_by_year}")

cfg = ReportConfig(
    output_root=Path("output"),
    title=_report_title,
    author="GREAT TEAM",
    targets=tuple(EDA_TARGETS),
    focus_predictor=FOCUS_PREDICTOR,
    focus_reference_level=FOCUS_REFERENCE_LEVEL,
    year_column=YEAR_COLUMN,
)
report_path = Path("output/report/report.html")
write_html(build_report(cfg), report_path)
print(f"Report written: {report_path.resolve()}")

Report written: /Users/andriszaguzovs/TheLibraryOfCode/RPE_petijums/dev/output/report/report.html


## 13. Outputs

Everything is saved on disk:

```
output/
├── dda/{figures,tables}/
├── missingness/{figures,tables}/
├── eda/{figures,tables}/
└── inferential/{figures,tables}/
```

Each plot is an individual `.svg`; each result table an individual `.csv`.


In [ ]:
from pathlib import Path
for p in sorted(Path(OUTPUT_ROOT).rglob('*')):
    if p.is_file():
        print(p)


output/.DS_Store
output/Icon
output/cleaning/cleaning_log.csv
output/cleaning/cleaning_summary.csv
output/dda/figures/Biopsijas_veids__bar.svg
output/dda/figures/EAU_riska_grupa__bar.svg
output/dda/figures/Gads__bar.svg
output/dda/figures/Gleason_grade_pēc_RPE__bar.svg
output/dda/figures/Gleason_grade_pēc_biopsijas__bar.svg
output/dda/figures/Klīniski_nozīmīgs_upgrade__bar.svg
output/dda/figures/Klīniski_nozīmīgs_upstage__bar.svg
output/dda/figures/Konkordance__bar.svg
output/dda/figures/PSA_blīvums__box.svg
output/dda/figures/PSA_blīvums__hist.svg
output/dda/figures/PSA_blīvums_grupēts__bar.svg
output/dda/figures/Pirmsoperācijas_PSA__box.svg
output/dda/figures/Pirmsoperācijas_PSA__hist.svg
output/dda/figures/Pirmsoperācijas_PSA_grupēts__bar.svg
output/dda/figures/Pirmsoperācijas_TNM_MDK__bar.svg
output/dda/figures/Pozitīvas_rezekcijas_līnijas__bar.svg
output/dda/figures/Priekšdziedzera_tilpums__box.svg
output/dda/figures/Priekšdziedzera_tilpums__hist.svg
output/dda/figures/Priekšdzied

## 14. NOTES — Why each statistical choice

Concise but detailed rationale for every formula used in this pipeline.
For each: **what it does**, **why chosen**, **what was rejected**.

---

### Schema inference (hybrid auto + override)

- **What.** Heuristic classification of each column into `continuous / count / ordinal / nominal / binary / datetime / id / text / skip` based on dtype, cardinality, value patterns.
- **Why.** Test selection downstream is kind-driven — a wrong kind silently picks the wrong test (e.g. treating Gleason 1–5 as `continuous` instead of `ordinal` swaps Spearman for MWU and loses interpretability of "per-grade increase").
- **Alternatives rejected.**
  - *Full auto-only*: brittle on clinical data where 0/1-coded ordinals look numeric.
  - *Manual ColSpec per column*: correct but tedious; you'd re-type 30+ specs per study.

---

### Duplicate auditing on normalized string keys

- **What.** Lowercase + strip + empty→NA on ID columns, then flag rows whose full key tuple is non-null and repeated.
- **Why.** Clinical IDs (`pk`, `year`) frequently have invisible whitespace or case drift across data-entry sessions. Naive `duplicated()` misses these.
- **Alternatives rejected.**
  - *Exact match*: under-detects.
  - *Fuzzy match (Levenshtein)*: over-detects, would falsely merge genuinely different patients.

---

### Mann–Whitney U for continuous/count vs binary outcome

- **What.** Non-parametric rank-sum test. H₀: P(X₁ > X₂) = ½. Two-sided.
- **Effect size.** Rank-biserial **r = |Z|/√N**, where Z is the large-sample normal approximation of U. Bounded 0–1, interpretable like Cohen's r (0.1 small, 0.3 medium, 0.5 large).
- **Why.**
  - Clinical continuous variables (PSA, vecums, days-to-surgery) are **almost never normal** — PSA in particular is heavily right-skewed.
  - MWU has ~95% efficiency vs t-test under normality and is far more robust under non-normality.
  - One test for the whole pipeline = no test-switching artifacts.
- **Alternatives rejected.**
  - *Welch's t-test always*: violates assumption on skewed data; inflates type-I error on small skewed samples.
  - *Auto Shapiro-Wilk switch (t if normal, MWU else)*: the normality test itself adds noise and its decision is sample-size dependent (always rejects normal at large N, never at small N) — produces worse calibration than just using MWU.
  - *Welch's t on log-transformed data*: works for PSA specifically but not generalizable to all continuous predictors in the pipeline.
- **Sensitivity.** When publishing, re-run Welch's t on log(PSA) as a sensitivity analysis — if direction and significance agree with MWU, you're robust.

---

### Spearman ρ for ordinal vs binary outcome

- **What.** Pearson correlation on the ranks of category codes vs the 0/1-encoded outcome.
- **Why.**
  - Preserves the **ordering** of ordinal predictors (Gleason 1<2<3<4<5, PIRADS 1<2<3<4<5, risk_group low<mid<high). χ² throws this away — it would only tell you "the distribution differs across levels", not "higher Gleason → more upgrades".
  - Yields a signed, scale-free effect size (ρ) that's directly publishable.
- **Alternatives rejected.**
  - *χ² on the ordinal × binary table*: ignores ordering, weaker power, no direction.
  - *Cochran-Armitage trend test*: equivalent to a linear-trend variant of χ² and gives p only — Spearman gives p **plus** a comparable ρ across all ordinal predictors.
  - *Kendall's τ*: similar info but slower on large N and no power advantage here.

---

### χ² (or Fisher exact) for nominal vs binary

- **What.** χ² of independence on the contingency table, **without Yates correction** (modern recommendation — Yates is overconservative). Switches to **Fisher exact** if the 2×2 table has any expected cell count < 5.
- **Why Fisher when expected<5.** χ²'s asymptotic distribution breaks down with small expected counts; Fisher's exact test conditions on the marginals and computes the exact hypergeometric p — correct at any sample size.
- **Effect size: Cramér's V** = √(χ²/(N·(min(r,c)−1))). Bounded 0–1, comparable across table shapes. For 2×2 tables we **also** report the odds ratio because clinicians read OR natively.
- **Alternatives rejected.**
  - *Yates-corrected χ²*: too conservative for modern computing — Fisher is exact and almost as fast.
  - *G-test (likelihood ratio)*: theoretically nicer for nested models but identical conclusions in 2-way tables; less familiar to clinical readers.
  - *Permutation χ²*: same answer as Fisher for 2×2, more expensive.

---

### Benjamini–Hochberg FDR correction, per target

- **What.** Sort p-values ascending; for rank i out of m, compute q_i = p_(i)·m/i; enforce monotonicity from the right; significance at q < α controls expected proportion of false discoveries at α.
- **Why per-target (not pooled across all targets).** Each outcome (upgrade, upstage, downgrade) is a **separate family** of hypotheses with its own scientific interpretation. Pooling them inflates the family size and over-corrects. This matches how clinical journals report multi-outcome studies.
- **Alternatives rejected.**
  - *Bonferroni*: controls family-wise error rate — far too conservative for a screening stage with 10+ predictors. Misses real signal.
  - *Holm-Bonferroni*: still FWER, marginally less conservative than Bonferroni but still much stricter than BH.
  - *Storey q-value*: estimates the null proportion adaptively; great when you have hundreds of tests but unstable at small m (you'll have <20 tests per target).
  - *No correction*: indefensible with ≥3 predictors per target — false discovery rate would be ~30%+.
- **Verified.** Output matches `statsmodels.stats.multitest.multipletests(method='fdr_bh')` exactly.

---

### MICE (Multiple Imputation by Chained Equations), m=10

- **What.** For each missing value: fit a regression of that column on all others using observed data, predict missing values, iterate until convergence. Repeat with m different random seeds to produce m plausible completed datasets.
- **Why multiple (not single).** Single imputation pretends the imputed values are known, so it **understates standard errors**. With m=10 imputations and Rubin pooling, the SEs honestly include imputation uncertainty.
- **Estimator: RandomForestRegressor.** Captures non-linear relationships (PSA × vecums × Gleason interactions) without you specifying them. Tolerates mixed numeric/categorical inputs.
- **Why m=10.** Rubin showed efficiency = (1 + fmi/m)^(-1) where fmi is fraction of missing info. At fmi ≈ 0.3 (typical clinical data), m=10 gives ~97% efficiency. m=5 is acceptable, m=20 is overkill.
- **Alternatives rejected.**
  - *Mean/median imputation*: distorts variance and any correlation involving the imputed column. Catastrophic for inferential SEs.
  - *Complete-case analysis*: throws away rows with any missingness — typically 20–50% data loss in clinical cohorts; introduces selection bias if missingness is MAR (which it usually is).
  - *Hot-deck imputation*: works for nominal-only data; weaker for mixed types.
  - *Bayesian model-based imputation (`mice` R package, Stan)*: gold standard but heavy infrastructure; sklearn's `IterativeImputer` is close enough for clinical screening.
- **Limitation.** Assumes data is **Missing At Random** (MAR) — missingness depends only on observed variables. For **MNAR** patterns (e.g. "PSA was missing because risk was low"), add explicit `<col>_missing` flags in section 6a so the model can use the missingness indicator itself as a predictor.

---

### Missingness flags

- **What.** Binary indicator columns `<col>_missing` added before imputation.
- **Why.** In clinical data, *that a value was missing* is often informative (e.g. PSA not measured because clinician judged it unnecessary). Including the flag in the regression lets the model separate "the value's effect" from "the act of measuring's effect".
- **Alternatives rejected.**
  - *Imputing without flags*: hides the MNAR mechanism.
  - *Dropping columns with high missingness*: throws away signal; missingness % is not a reliable filter for clinical utility.

---

### Variance Inflation Factor (VIF) pruning, threshold = 5

- **What.** For each predictor x_j, VIF = 1/(1 − R²_j), where R²_j is from regressing x_j on all other predictors. Iteratively drop the column with the highest VIF until all ≤ 5.
- **Why.** Logistic regression with collinear predictors produces enormous standard errors and unstable coefficients ("model can't tell whether PSA or PSA-density is doing the work"). VIF > 5 ⇔ R²_j > 0.80 ⇔ severe multicollinearity.
- **Why threshold = 5** (not 10). VIF=10 is the classical statistics teaching threshold but for clinical regression with modest N (<500), 5 is the modern recommendation (Vatcheva 2016, O'Brien 2007).
- **Alternatives rejected.**
  - *Pairwise Pearson correlation > 0.8*: catches only 2-variable collinearity; misses 3-way (e.g. a = b + c).
  - *Lasso regularization*: would drop collinear features automatically but **biases coefficients** toward zero — bad for inference (you want unbiased OR estimates). Lasso is for prediction, not inference.
  - *Ridge / Elastic Net*: same problem — shrinks coefficients, distorts ORs.
  - *PCA / partial-least-squares*: components are uninterpretable clinically.

---

### Multivariable logistic regression

- **What.** Per target, one binary logistic model with all surviving predictors. Continuous z-scored (so OR is per-SD increase), ordinals kept as numeric codes, nominals one-hot with drop_first.
- **Why.** Univariate screening (section 10) ignores confounding — Gleason can show up "significant" purely because it correlates with PSA. Multivariable estimates the **adjusted** effect of each predictor holding the others constant.
- **Why this design encoding.**
  - *z-score continuous*: ORs comparable across predictors; one "unit" = one SD.
  - *Ordinal as numeric code*: assumes linear log-odds across levels (parsimonious; standard for Gleason/PIRADS in urology papers). The alternative is one-hot with drop_first, which uses more degrees of freedom and is only worth it if the trend is clearly non-monotonic — check the EDA plots first.
  - *Nominal one-hot drop_first*: avoids the dummy variable trap (perfect collinearity with intercept).
- **Alternatives rejected.**
  - *Univariate-only pipeline*: misleading because of confounding.
  - *Stepwise selection (forward/backward)*: notorious for unstable selection, inflated significance, and irreproducibility. Modern guidance (Harrell, Steyerberg) is: don't.
  - *Random forest / XGBoost*: better predictive accuracy but no clinical OR with CI to report.
  - *Penalized regression (Firth, Lasso, Ridge)*: useful with extreme separation or n<<p but biases the OR estimates — defeats the inferential purpose.
  - *Bayesian logistic with weakly informative priors*: cleaner for tiny samples and would give credible intervals — but you'd need to defend prior choice in the manuscript.

---

### Rubin's rules with Barnard–Rubin degrees of freedom

- **What.** Across the m=10 imputed-frame fits, for each coefficient:
  - θ̄ = mean of the m point estimates
  - within-imp variance Ū = mean of the m squared SEs
  - between-imp variance B = sample variance of the m estimates
  - total variance T = Ū + (1 + 1/m)·B
  - pooled SE = √T
  - degrees of freedom (Barnard–Rubin):
    df = (m−1)·(1 + Ū/((1+1/m)·B))²
  - p-value from t-distribution with that df; 95% CI = θ̄ ± t_{0.975, df}·SE
- **Why.** Rubin's rules are the **only** statistically valid way to combine results across multiple imputations. The total variance T splits into "within" (each model's uncertainty) and "between" (uncertainty due to missing data) — they're not interchangeable.
- **Why Barnard–Rubin df (not the original Rubin 1987 df).** Original Rubin df → ∞ when between-variance is small, which is wrong when m is small. Barnard–Rubin (1999) is a small-sample correction that's now the standard (R `mice` uses it, SAS PROC MIANALYZE uses it).
- **Alternatives rejected.**
  - *Picking the "best" imputation*: defeats the purpose of multiple imputation entirely.
  - *Average the imputed datasets first, then fit once*: produces correct point estimates but **wrong SEs** (the between-variance is invisible).
  - *Use the within-variance only*: ignores imputation uncertainty — false confidence.
- **Verified.** With zero between-variance, our pooler returns SE equal to the single-fit SE; with non-zero between, it correctly inflates SE and produces a finite small-sample df (e.g. m=5, modest B → df ≈ 22).

---

### Why log-scale x-axis on forest plots

- ORs are multiplicative (OR=2 and OR=0.5 are equal-and-opposite effects). On a linear axis they look asymmetric; on log scale they're symmetric around OR=1, which is the correct visual.

---

### What this pipeline deliberately does NOT do

- **No machine-learning prediction** (no train/test split, no AUC, no calibration). This is an **association/inference** pipeline, not a prediction pipeline. If you later want a predictive model (e.g. nomogram for upgrade risk), that's a separate workflow with cross-validation, calibration plots, decision-curve analysis.
- **No causal inference** (no DAGs, no IPTW, no instrumental variables). All effects here are **statistical associations** adjusted for the included covariates — they are *not* causal effects. Manuscript wording must say "associated with", never "causes".
- **No survival/time-to-event analysis.** Targets here are binary (upgrade yes/no). If you later care about *time to biochemical recurrence*, you'd need Cox regression — a separate module.

---

### Sanity-check checklist before submitting results

1. Print `schema_summary(schema)` — every ordinal has correct `ordered_levels`?
2. After MICE, `imputed_frames[0].isna().sum().sum()` == 0 for predictor columns?
3. `inf_results['n_models']` ≈ m for all predictors (means the model converged on every imputation)?
4. Forest plot ORs and EDA univariate effects agree in **direction** (sign)? If they flip, you have confounding worth discussing.
5. For each FDR-significant univariate result, check the corresponding plot in `output/eda/figures/` — is the pattern visually credible or driven by 2–3 outliers?
